# Grant FP Classifier — INFERENCE (scoring) run
**SoOI 2026 — score the production dataset, drop false-positive matches**

This is the **scoring** twin of the evaluation notebook. The eval notebook measures the prompt on the
labelled test set (`label` column). This one runs the **validated** prompt over the unlabelled production
data to assign `predicted` to every (grant × OI) row, then drops the FALSE matches.

**Pipeline position:** `01 → 02 → FP filter (this notebook) → award classifier → 03`.
Runs on `deduplicated.csv` (not `enriched.csv`) so dropping FALSE rows happens *before* `03` computes
cluster-level fields (`MULTI_RECIPIENT`, `DASH_AMT`, `AWARD_SOLUTION_CATEGORY`) — otherwise those go stale.

**Before running:** paste your final validated `SYSTEM_PROMPT` + `FEW_SHOT_EXAMPLES` from the eval
notebook (cell 13) into the prompt cell below, so the eval notebook stays the single source of truth.

## 1. Setup

In [ ]:
!pip install anthropic pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 9.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import anthropic, pandas as pd, time, datetime
from google.colab import userdata
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
print("✓ Setup complete")

✓ Setup complete


## 2. Configuration

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────
BASE = '/content/drive/Shareddrives/IOI Team Drive/Research/Projects_Research/Current_Projects_Research/2026_State_of_OI/02_Research/Grant Funding'
DATA_PATH  = f'{BASE}/deduplicated.csv'              # input: output of notebook 02
OUT_SCORED = f'{BASE}/deduplicated_scored.csv'       # every row + `predicted`
OUT_CLEAN  = f'{BASE}/deduplicated_clean.csv'        # FALSE dropped -> feeds the award classifier, then 03

MODEL = 'claude-sonnet-4-6'        # final scoring run uses Sonnet (Haiku was for prompt iteration)
OUTPUT_REASONING = False           # True -> also store a 1-line reason per row (slower, more tokens)
API_DELAY = 0.1                    # seconds between calls; raise to 0.5 if rate-limited
CHECKPOINT_EVERY = 200             # save partial progress to OUT_SCORED every N rows

# Auto-NEEDS_REVIEW: rows with no usable context are flagged WITHOUT an API call (no wasted call,
# no model guessing from nothing). Flagged if title+description are both empty, or the combined
# text has fewer than THIN_MIN_WORDS words. Set THIN_MIN_WORDS = 0 to disable.
THIN_MIN_WORDS = 4
# ────────────────────────────────────────────────────────────────────────────
print('Model:', MODEL, '| reasoning:', OUTPUT_REASONING, '| thin-context cutoff:', THIN_MIN_WORDS, 'words')

Model: claude-sonnet-4-6 | reasoning: False | thin-context cutoff: 4 words


## 3. Load & map columns + auto-flag thin rows

In [ ]:
df = pd.read_csv(DATA_PATH, dtype=str)

# Map enriched/deduplicated column names -> what the classifier functions expect.
# record_uid (row-unique) is the join key, NOT GRANT_ID (not unique after multi-OI / synthetic IDs).
df['oi_name']           = df['OI']
df['grant_title']       = df['TITLE']
df['grant_description']  = df['DESCRIPTION']
assert 'record_uid' in df.columns, "expected record_uid as the unique key"

def _clean(x):
    s = str(x or '').strip()
    return '' if s.lower() in ('', 'nan', 'none') else s

def is_thin(r):
    t, d = _clean(r['grant_title']), _clean(r['grant_description'])
    if t == '' and d == '': return True
    return len((t + ' ' + d).split()) < THIN_MIN_WORDS

df['predicted'] = ''
df['reasoning'] = None
if THIN_MIN_WORDS > 0:
    thin_mask = df.apply(is_thin, axis=1)
    df.loc[thin_mask, 'predicted'] = 'NEEDS_REVIEW'
    print(f'Auto-flagged NEEDS_REVIEW (no usable context, no API call): {int(thin_mask.sum())}')

print(f'Total rows: {len(df)} | to classify via API: {int((df.predicted=="").sum())}')

Auto-flagged NEEDS_REVIEW (no usable context, no API call): 37
Total rows: 3560 | to classify via API: 3523


## 4. Validated prompt  — PASTE FROM EVAL NOTEBOOK (cell 13)

Copy your final `SYSTEM_PROMPT` and `FEW_SHOT_EXAMPLES` verbatim from the evaluation notebook so the
two stay identical. Do not re-tune here — this run uses the prompt you already validated.

In [ ]:
# ===== PASTE: SYSTEM_PROMPT and FEW_SHOT_EXAMPLES from the eval notebook cell 13 =====
SYSTEM_PROMPT = """You are classifying whether a grant record is a TRUE MATCH or FALSE POSITIVE for a given open infrastructure (OI).

TRUE = the OI is explicitly named as something the project will use,
connect to, deposit into, publish through, promote, or integrate —
even if it is one of several tools named, or a minor part of the project.

This includes:
- Directly funding or sustaining the OI
- Using it as a tool, platform, or repository (depositing data, publishing
  preprints, accessing records, submitting manuscripts, etc.)
- Integrating it as a component of what the project builds
- Disseminating outputs through it (preprints, datasets, code, results)
- Promoting datasets or tools on it alongside other platforms
- Linking to or connecting with it as one of several named identifiers or services

FALSE = the OI is not explicitly named with intent to use, or the term
matched something unrelated:
- The matched word or acronym refers to something else entirely (a gene,
  a common word, a different organisation, a computer science term)
- The OI is cited only as a landscape example of what exists
  ("activities such as X", "providers such as X, Y, and Z") with no
  commitment to use it
- The OI is named only as a comparison or inspiration, not as something
  the project will actually use ("on the model of X", "inspired by X",
  "similar to X")
- The OI name is used as a generic noun ("a dataverse", "a commons")
  rather than referring to the specific named platform
- The grant is about the same topic as the OI but does not explicitly
  name it as something the project will use
- The OI is acknowledged as prior work or community context the project
  builds upon intellectually, without committing to use the platform
  ("builds upon the success of X", "extends the work of X",
  "motivated by X")

Important: topical relevance alone is not sufficient for TRUE. A grant
about persistent identifiers, open science, or data repositories is not
a TRUE match for a specific OI unless that OI is explicitly named with
intent to use it.

Respond with exactly one word: TRUE, FALSE, or UNCERTAIN.
Use UNCERTAIN when the grant mentions the OI but it is genuinely unclear
whether the relationship constitutes active use or merely acknowledgement."""

FEW_SHOT_EXAMPLES = [
    (
        "Fulcrum",
        "4.48 Psychosis: A National Theatre Revival",
        "Delivered with 4 national organisational partners and 4 academic "
        "researchers, 4.48 will grow debate and research, with its fulcrum "
        "a revival of 4.48 PSYCHOSIS delivered alongside an extended "
        "auxiliary event programme.",
        "FALSE"
        # 'fulcrum' is a common noun, not a reference to the OI
    ),
    (
        "ROR",
        "Characterisation of ROR1 and ROR2 receptor tyrosine kinases in "
        "breast cancer progression",
        "ROR1 and ROR2 are receptor tyrosine kinase-like orphan receptors "
        "implicated in oncogenesis. This study investigates their expression "
        "patterns and downstream signalling in triple-negative breast cancer "
        "cell lines.",
        "FALSE"
        # ROR is a gene family acronym here, not the Research Organisation Registry
    ),
    (
        "RRIDs",
        "Collaborative Research: FAIR Facilities and Instruments: Enabling "
        "transparency, reproducibility, and equity through persistent identifiers",
        "This research coordination network will advance the standardization "
        "and adoption of Persistent Identifiers (PIDs) for research facilities "
        "and instruments nationally. PIDs are widely seen as foundational to "
        "open science. While many categories of PIDs have come a long way "
        "(examples: DOIs for publications, ORCIDs for researchers), this "
        "project is focused on advancing PIDs for research facilities and "
        "instruments. This project builds on activities already being "
        "undertaken such as RRIDs (Research Resource Identifiers), and will "
        "convene a series of workshops and stakeholder engagements to better "
        "understand the field and develop principles of design for new PIDs.",
        "FALSE"
        # RRIDs cited as a landscape example of existing activity — grant
        # is about PIDs generally, not using or funding RRIDs specifically.
        # Topical relevance alone is not sufficient.
    ),
    (
        "Dataverse",
        "CC* Data: ImPACT - Infrastructure for Privacy-Assured compuTations",
        "To enable end-to-end data flows, ImPACT builds upon prior investments "
        "and enabling cyberinfrastructure technologies like Dataverse, CyVerse, "
        "and the Open Resource Control Architecture (ORCA) control software.",
        "FALSE"
        # Dataverse is named in a landscape list of enabling technologies —
        # no explicit commitment to use it
    ),
    (
        "Dataverse",
        "PIRE: Deeply Decarbonizing Global Industrial Supply Chains",
        "The outcome of this research will be the development of a novel "
        "global dataverse and the advancement of machine learning approaches "
        "to examine GHG emissions drivers for industrial plants.",
        "FALSE"
        # 'dataverse' used as a generic noun meaning a data collection,
        # not a reference to the Dataverse platform
    ),
    (
        "arXiv",
        "Multimessenger astronomy following neutron star merger GW170817",
        "The merger of neutron stars GW170817 has been the best observed "
        "astronomical event in recorded history. Close to 150 preprints "
        "dedicated to the merger appeared on the arXiv during the two days "
        "after the publication of the LIGO result in October 2017.",
        "TRUE"
        # Researchers actively published preprints on arXiv as part of the work
    ),
    (
        "arXiv",
        "AF: Small: Streaming Complexity of Constraint Satisfaction Problems",
        "Progress from the project will be reported on public domain sites "
        "like the arxiv (www.arxiv.org).",
        "TRUE"
        # Explicit commitment to disseminate outputs via arXiv counts as TRUE
        # even as a minor dissemination mention
    ),
    (
        "Dataverse",
        "Cooperative Congressional Election Study 2016",
        "The data produced by this project will be a 2016 Common Content "
        "dataset and will be available on the CCES Dataverse website.",
        "TRUE"
        # Project data is actively deposited and hosted on Dataverse
    ),
    (
        "Pangeo",
        "Climate Process Team on Ocean Transport and Eddy Energy",
        "Newly generated datasets and parameterizations will be promoted on "
        "many platforms (e.g., Pangeo, GitHub), which can be used by the "
        "whole climate community.",
        "TRUE"
        # Explicitly named with intent to promote outputs there — counts as
        # TRUE even as one of two platforms in a parenthetical
    ),
    (
        "PubPub",
        "Supporting the Knowledge Futures Group community publishing platform",
        "To support the growth of the Knowledge Futures Group's PubPub "
        "community publishing platform, enabling open access publishing "
        "infrastructure for academic communities.",
        "TRUE"
        # Grant directly funds PubPub
    ),
]
# =====================================================================================
assert SYSTEM_PROMPT.strip() and not SYSTEM_PROMPT.startswith('...PASTE'), "Paste the validated SYSTEM_PROMPT"
assert len(FEW_SHOT_EXAMPLES) > 0, "Paste the validated FEW_SHOT_EXAMPLES"
print(f'Prompt loaded | {len(FEW_SHOT_EXAMPLES)} few-shot examples')

Prompt loaded | 10 few-shot examples


## 5. Classify functions

In [ ]:
def build_messages(oi_name, grant_title, grant_description, include_reasoning=False):
    messages = []
    for ex_oi, ex_title, ex_desc, ex_label in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user", "content": f"Infrastructure: {ex_oi}\nGrant title: {ex_title}\nGrant description: {ex_desc}"})
        messages.append({"role": "assistant", "content": ex_label})
    suffix = "\n\nRespond with TRUE or FALSE, then on a new line give a brief reason (1 sentence)." if include_reasoning else ""
    messages.append({"role": "user", "content": f"Infrastructure: {oi_name}\nGrant title: {grant_title}\nGrant description: {grant_description}{suffix}"})
    return messages

def classify_grant(row, include_reasoning=False):
    resp = client.messages.create(
        model=MODEL, max_tokens=(300 if include_reasoning else 100),
        system=SYSTEM_PROMPT,
        messages=build_messages(row['oi_name'], row['grant_title'], row['grant_description'], include_reasoning))
    raw = resp.content[0].text.strip()
    first = raw.split()[0].upper().rstrip('.:,') if raw else ''
    label = 'TRUE' if first.startswith('TRUE') else 'FALSE' if first.startswith('FALSE') else 'PARSE_ERROR'
    reasoning = raw.split('\n', 1)[1].strip() if (include_reasoning and '\n' in raw) else None
    return label, reasoning, raw

In [ ]:
# Smoke test: first 5 rows that aren't auto-flagged
print("=== Smoke test ===")
for _, row in df[df.predicted == ''].head(5).iterrows():
    label, reasoning, raw = classify_grant(row, OUTPUT_REASONING)
    print(f"OI: {row['oi_name']} | Predicted: {label} | Title: {str(row['grant_title'])[:70]}")
    if reasoning: print(f"   reason: {reasoning}")

=== Smoke test ===
OI: vivo | Predicted: FALSE | Title: New Patient-derived MicroTumor Culture/Assay System
OI: fulcrum | Predicted: FALSE | Title: Substrate Capture in Hsp70 Catalyzed by the Hsp40 J-domain
OI: argos | Predicted: FALSE | Title: University of Greenwich and Argos Limited
OI: crossref | Predicted: TRUE | Title: Make Data Count: A Central Corpus for All Data Citations  
OI: data citation corpus | Predicted: TRUE | Title: Make Data Count: A Central Corpus for All Data Citations  


## 6. Run scoring (checkpointed)

In [ ]:
todo = df[df.predicted == ''].index.tolist()
print(f'Classifying {len(todo)} rows with {MODEL}...')
start = time.time()
for n, i in enumerate(todo, 1):
    try:
        label, reasoning, raw = classify_grant(df.loc[i], OUTPUT_REASONING)
    except Exception as e:
        label, reasoning = 'API_ERROR', str(e)
    df.at[i, 'predicted'] = label
    df.at[i, 'reasoning'] = reasoning
    time.sleep(API_DELAY)
    if n % CHECKPOINT_EVERY == 0 or n == len(todo):
        df.to_csv(OUT_SCORED, index=False)
        print(f'  {n}/{len(todo)}  ({time.time()-start:.0f}s)  checkpointed')
df.to_csv(OUT_SCORED, index=False)
print('\n✓ Done. Prediction counts:'); print(df.predicted.value_counts(dropna=False))
errs = int((df.predicted.isin(['PARSE_ERROR','API_ERROR'])).sum())
if errs: print(f'⚠ {errs} parse/API errors — inspect before filtering')

Classifying 3523 rows with claude-sonnet-4-6...
  200/3523  (316s)  checkpointed
  400/3523  (639s)  checkpointed
  600/3523  (977s)  checkpointed
  800/3523  (1330s)  checkpointed
  1000/3523  (1653s)  checkpointed
  1200/3523  (1971s)  checkpointed
  1400/3523  (2292s)  checkpointed
  1600/3523  (2597s)  checkpointed
  1800/3523  (2917s)  checkpointed
  2000/3523  (3226s)  checkpointed
  2200/3523  (3530s)  checkpointed
  2400/3523  (3871s)  checkpointed
  2600/3523  (4212s)  checkpointed
  2800/3523  (4543s)  checkpointed
  3000/3523  (4902s)  checkpointed
  3200/3523  (5238s)  checkpointed
  3400/3523  (5595s)  checkpointed
  3523/3523  (5804s)  checkpointed

✓ Done. Prediction counts:
predicted
FALSE           2332
TRUE            1111
PARSE_ERROR       74
NEEDS_REVIEW      37
API_ERROR          6
Name: count, dtype: int64
⚠ 80 parse/API errors — inspect before filtering


## 7. Filter FALSE → clean file, and review high-value drops

In [ ]:
# Keep everything that isn't a confident FALSE (TRUE + NEEDS_REVIEW + UNCERTAIN are retained for review/use).
df = pd.read_csv(OUT_SCORED, dtype=str)   # or your reviewed filename, if you saved it separately
print(df.predicted.value_counts(dropna=False))

drop_mask = df.predicted == 'FALSE'
clean = df[~drop_mask].drop(columns=['oi_name','grant_title','grant_description'])   # shed the working aliases
clean.to_csv(OUT_CLEAN, index=False)
print(f'Scored {len(df)} -> kept {len(clean)} (dropped {int(drop_mask.sum())} FALSE) -> {OUT_CLEAN}')
print('Retained NEEDS_REVIEW:', int((df.predicted=="NEEDS_REVIEW").sum()))

# Don't auto-trust FALSE on big-money rows: eyeball the largest before they're gone.
amt = pd.to_numeric(df.get('AMOUNT'), errors='coerce')
big_false = df[drop_mask].assign(_amt=amt[drop_mask]).sort_values('_amt', ascending=False)
print('\n=== Largest FALSE-flagged awards (review before trusting the drop) ===')
cols = [c for c in ['record_uid','oi_name','FUNDER','AMOUNT','CURRENCY','grant_title'] if c in df.columns]
pd.set_option('display.max_colwidth', 80)
display(big_false[cols].head(20))

predicted
FALSE    2385
TRUE     1175
Name: count, dtype: int64
Scored 3560 -> kept 1175 (dropped 2385 FALSE) -> /content/drive/Shareddrives/IOI Team Drive/Research/Projects_Research/Current_Projects_Research/2026_State_of_OI/02_Research/Grant Funding/deduplicated_clean.csv
Retained NEEDS_REVIEW: 0

=== Largest FALSE-flagged awards (review before trusting the drop) ===


,record_uid,oi_name,FUNDER,AMOUNT,CURRENCY,grant_title
3140,openaire::ANR-10-AIRT-0008::open policy finder (formerly sherpa services),open policy finder (formerly sherpa services),[French National Research Agency (ANR)],132546000,EUR,NaN
2359,HHS::P51OD011133::biorxiv,biorxiv,Department of Health and Human Services,122234337,USD,NaN
2782,openaire::807081::dspace,dspace,[European Commission],113185000,EUR,Systems ITD
243,openalex::14102018::dspace,dspace,[Japan Society for the Promotion of Science],103870000,JPY,Ultra-High-speed Sensory-Motor System Based on Distributed Network Architecture
591,DOD::FA86501325503::fulcrum,fulcrum,NaN,74687361,USD,NaN
1295,openaire::945535::dspace,dspace,[European Commission],62623700,EUR,SYSTEMS ITD GAM 2020
2406,openalex::23240098::open policy finder (formerly sherpa services),open policy finder (formerly sherpa services),[Japan Society for the Promotion of Science],49400000,JPY,Creation of standard values that account for the relationship of environment...
2964,openalex::16205025::thoth open metadata,thoth open metadata,[Japan Society for the Promotion of Science],49010000,JPY,Creation of Chiral Organic Photochromic Dyes Applicable to Ultra-high Densit...
2313,openalex::22249031::public utility data liberation (pudl),public utility data liberation (pudl),[Japan Society for the Promotion of Science],46670000,JPY,Researches using iPS cells on intractable respiratory diseases to elucidate ...
285,openalex::22251013::journal article tag suite,journal article tag suite,[Japan Society for the Promotion of Science],43940000,JPY,Interdisciplinary study of the process of domesticasion and pastoralism in t...


## Notes
- `OUT_CLEAN` (FALSE dropped, NEEDS_REVIEW kept) is the input to the **award classifier**, then `03`.
- `predicted` is now your filter, not ground truth — the cleaned set still carries the classifier's
  known residual error (~0.86 FALSE-F1). State that band in the methodology note.
- To re-trust a wrongly-dropped big award: set its `predicted` to `TRUE` in `deduplicated_scored.csv`
  and re-run cell 7 (or just keep it — the clean file is regenerated from `predicted`).